**Re-define virality with new data as a threshold of reposts on the post x amount of hours after creation.**

In [3]:
import pandas as pd
import numpy as np
from kneed import KneeLocator
import duckdb
import json
import tarfile
import matplotlib.pyplot as plt
from pathlib import Path
import sys


In [ ]:
BASE_PATH = Path.cwd().parent
sys.path.append(str(BASE_PATH))
root_posts = pd.read_parquet(BASE_PATH/"datasets/bluesky_cascade/root_posts.parquet")
print(root_posts.shape)

In [ ]:
# Compare distributions of reposts after x hours
diff_hours = [3,4,6,12,24]
repost_dist_after_each_num_hours = {}

for hours in diff_hours:
    repost_dist_after_x_hours = (
    root_posts[f"repost_{hours}h"]
    .value_counts()
    .sort_index()
    .reset_index()
)
    repost_dist_after_each_num_hours[hours] = repost_dist_after_x_hours

# Plot
fig, ax = plt.subplots(figsize=(20, 6))

ax.violinplot(
    [repost_dist_after_each_num_hours[h] for h in diff_hours],
    positions=range(1,25),
    showmedians=True
)

ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost count")
ax.set_title("Distribution of reposts received within X hours of creation")
ax.set_xticks(range(1, 25, 2))
plt.tight_layout()
plt.show()

Analyse the knees for each number of hours

In [ ]:
knee_results = []

for hours in diff_hours:
    try:
        kneedle = KneeLocator(
            repost_dist_after_each_num_hours[hours]["repost_count"],
            repost_dist_after_each_num_hours[hours]["post_count"],
            curve="convex",
            direction="decreasing"
        )
        knee = kneedle.knee
    except Exception as e:
        knee = None

    knee_results.append({"hours": hours, "knee": knee})
    print(f"Hour {hours:2d}: knee at {knee} reposts")

knee_df = pd.DataFrame(knee_results)

# Plot knee over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(knee_df["hours"], knee_df["knee"], marker="o")
ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost threshold (knee)")
ax.set_title("Virality threshold (knee of repost distribution) over time")
ax.set_xticks(range(1, 25, 2))
plt.tight_layout()
plt.show()

Analyse percentiles for each time period

In [ ]:
percentile_values = [0.9, 0.95, 0.98, 0.99, 0.999, 0.9999, 1]
percentile_results = []

for hours in diff_hours:
    row = {"hours": hours}
    counts = repost_dist_after_each_num_hours[hours]["repost_count"]
    for p in percentile_values:
        row[f"p{p}"] = counts.quantile(p)
    percentile_results.append(row)
    print(f"Hour {hours}: " + ", ".join(f"p{p}={row[f'p{p}']:.1f}" for p in percentile_values))

percentile_df = pd.DataFrame(percentile_results)

fig, ax = plt.subplots(figsize=(12, 5))
for p in percentile_values:
    ax.plot(percentile_df["hours"], percentile_df[f"p{p}"], marker="o", label=f"p{p}")
ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost count threshold")
ax.set_title("Repost distribution percentiles over time")
ax.legend()
plt.tight_layout()
plt.show()

**Define virality based on TI and S z-scores**